# Merge Temporal Features

| **Feature Store Type** | **Path** | **Description** |
|---|---|---|
| Cleaned Stock Market Features | `stock_features.csv`| Mean Log price for the month, Volatility and MOM Return |
| Cleaned Macroeconomic Features | `sg_macro_features_monthly.csv` | Inflation, unemployment, interest rates, GDP, Housing resale |
| Cleaned COE Features | `coe_features_monthly.csv` | bids_received, quota and quota_premium split by COE type for every month |

In [26]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 400)
ROOT = '/Users/keltonn/Desktop/Projects/housing_rent_price/final_report'
TEMPORAL = f'{ROOT}/data/features/cleaned/temporal_features'
OUTPUT = TEMPORAL                                           # the audit tables are saved next to the temporal features they describe

START_YEAR = 2021
N_MONTHS = 67                                               # months 0..66; test ends at month 66 (2026-07)


def month_index(year, month):
    """Months since 2021-01 (= 0)."""
    return (year - START_YEAR) * 12 + month - 1


def month_label(index):
    return f'{START_YEAR + int(index) // 12}-{int(index) % 12 + 1:02d}'


print('project root:', ROOT)

project root: /Users/keltonn/Desktop/Projects/housing_rent_price/final_report


## 1. Load the three temporal sources, one row per month

In [38]:
def load_by_year_month(path):
    """Stock and COE files carry RENT_APPROVAL_YEAR / RENT_APPROVAL_MONTH columns."""
    df = pd.read_csv(path)
    df.index = month_index(df['RENT_APPROVAL_YEAR'], df['RENT_APPROVAL_MONTH'])
    return df


def load_macro(path):
    """The macro file carries a single RENT_APPROVAL_DATE column instead."""
    df = pd.read_csv(path)
    date = pd.to_datetime(df.pop('RENT_APPROVAL_DATE'))
    df.index = month_index(date.dt.year, date.dt.month)
    return df


sources = {'stock': load_by_year_month(f'{TEMPORAL}/stock_features.csv'),
           'macro': load_macro(f'{TEMPORAL}/sg_macro_features_monthly.csv'),
           'coe': load_by_year_month(f'{TEMPORAL}/coe_features_monthly.csv').iloc[:67]}

for name, df in sources.items():
    assert df.index.is_unique, f'{name}: duplicate months'
    print(f'{name:16s} {df.shape[1]:4d} features, months {df.index.min()}..{df.index.max()} ({len(df)} rows)')

stock             233 features, months 0..66 (67 rows)
macro               6 features, months 0..66 (67 rows)
coe                14 features, months 0..66 (67 rows)


In [39]:
sources['coe'].columns

Index(['RENT_APPROVAL_YEAR', 'RENT_APPROVAL_MONTH', 'coe_a_bids_received', 'coe_a_quota', 'coe_a_quota_premium', 'coe_b_bids_received', 'coe_b_quota', 'coe_b_quota_premium', 'coe_c_bids_received',
       'coe_c_quota', 'coe_c_quota_premium', 'coe_e_bids_received', 'coe_e_quota', 'coe_e_quota_premium'],
      dtype='object')

## 2. Coverage of every feature

Check the coverage for every feature. We drop the features that do not have full coverage over the entire train/test/validation period.

For each feature: the first and last month with a value, the number of missing months, and whether it is a constant. `full` = no missing month (month 0 is
skipped for month-over-month returns). A month absent from a file counts as missing, and months beyond 66 are ignored (the COE file has one extra month).

In [43]:
STOCK_KINDS = 'log_price_mean|mom_return|return_volatility'

def feature_coverage(sources, n_months=N_MONTHS):
    rows = []
    for name, df in sources.items():
        df = df.reindex(range(n_months))
        for column in df:
            check = df[column].iloc[1:] if column.endswith('_mom_return') else df[column]
            valid = df[column].dropna().index
            rows.append({'source': name, 'feature': column,
                         'first_month': valid.min() if len(valid) else np.nan,
                         'last_month': valid.max() if len(valid) else np.nan,
                         'n_missing': int(check.isna().sum()),
                         'constant': df[column].nunique() <= 1,
                         'full': bool(check.notna().all())})
    coverage = pd.DataFrame(rows)
    is_stock = coverage['source'] == 'stock'
    coverage['kind'] = np.where(is_stock, coverage['feature'].str.extract(rf'({STOCK_KINDS})$')[0], 'level')
    coverage['stock'] = np.where(is_stock, coverage['feature'].str.replace(rf'^stk_|_({STOCK_KINDS})$', '', regex=True), '')
    return coverage


coverage = feature_coverage(sources)

print(coverage.groupby(['source', 'kind'])['full'].agg(['sum', 'size']).rename(columns={'sum': 'full', 'size': 'total'}))
print(f"\nfull in every month: {int(coverage['full'].sum())} of {len(coverage)} features")
print('constant features:', coverage.loc[coverage['constant'], 'feature'].tolist())

                          full  total
source kind                          
coe    level                14     14
macro  level                 6      6
stock  log_price_mean       53     77
       mom_return           53     77
       return_volatility    53     77

full in every month: 181 of 253 features
constant features: []


## 3. Economy features (macro and COE)

In [44]:
coverage[coverage['source'] != 'stock'][['source', 'feature', 'first_month', 'last_month', 'n_missing', 'constant', 'full']].reset_index(drop=True)

,source,feature,first_month,last_month,n_missing,constant,full
0,macro,MACRO_CPI_EX_ACCOM_YOY,0,66,0,False,True
1,macro,MACRO_SORA_3M_PCT,0,66,0,False,True
2,macro,MACRO_REAL_GDP_YOY,0,66,0,False,True
3,macro,MACRO_UNEMPLOYMENT_SA_PCT,0,66,0,False,True
4,macro,MACRO_HDB_RESALE_RPI_YOY,0,66,0,False,True
5,macro,MACRO_PRIVATE_RENT_RPI_YOY,0,66,0,False,True
6,coe,RENT_APPROVAL_YEAR,0,66,0,False,True
7,coe,RENT_APPROVAL_MONTH,0,66,0,False,True
8,coe,coe_a_bids_received,0,66,0,False,True
9,coe,coe_a_quota,0,66,0,False,True


## 4. Stocks

A stock's three features (`log_price_mean`, `mom_return`, `return_volatility`) have the same coverage, so stocks are listed rather than columns. The first month of the
stocks that do **not** exist throughout (they start late) is shown so it is clear what is dropped.

In [45]:
stock_coverage = coverage[coverage['source'] == 'stock']
assert (stock_coverage.groupby('stock')['full'].nunique() == 1).all()    # the 3 features of a stock agree

per_stock = (stock_coverage.groupby('stock')
             .agg(first_month=('first_month', 'min'), last_month=('last_month', 'max'),
                  n_missing=('n_missing', 'max'), full=('full', 'all')))
per_stock['first_month_date'] = per_stock['first_month'].map(month_label)

print(f"{int(per_stock['full'].sum())} of {len(per_stock)} stocks exist throughout\n")
print('EXIST THROUGHOUT:')
print(', '.join(per_stock.index[per_stock['full']]))
print('\nDO NOT (start late or have gaps):')
per_stock[~per_stock['full']].sort_values('first_month')[['first_month_date', 'last_month', 'n_missing']]

55 of 79 stocks exist throughout

EXIST THROUGHOUT:
RENT_APPROVAL_MONTH, RENT_APPROVAL_YEAR, ascendas_reit, boc_aviation, capitaland_ascott_trust, capitaland_mall_trust, city_developments, cmon, comfortdelgro, dbs_group, flex, frasers_logistics_industrial_trust, genting_singapore, golden_agri_resources, grab_holdings, haw_par, hotel_properties_limited, hph_trust_hutchison_port, ifast_corporation, igg_inc, jardine_cycle_carriage, kenon_holdings, keppel, keppel_reit, kulicke_and_soffa_industries, lightinthebox_holding, mapletree_commercial_trust, mapletree_industrial_trust, mapletree_logistics_trust, moneyhero, netlink_trust, ocbc_bank, olam, pacific_century_regional_developments, penguin_international_limited, riverstone_holdings, sats, sea_limited, seatrium, sembcorp, sheng_siong_group, sia_engineering_company, singapore_airlines, singapore_exchange, singapore_land, singapore_post, singtel, st_engineering, suntec_reit, uob, uol_group, venture_corporation, wave_life_sciences, wilmar_int

,first_month_date,last_month,n_missing
stock,,,
karooooo,2021-04,66,3
bitdeer_technologies_group,2021-07,66,6
guardforce_ai,2021-09,66,9
capitaland_investment_limited,2021-09,66,8
bitfufu,2021-12,66,11
euda_health,2021-12,66,11
gcl_global_holdings_ltd,2022-04,66,15
genius_group,2022-04,66,15
super_hi_international,2022-12,66,24


## 5. The final list of features that exist throughout

Saved for the modelling pipeline. The original pipeline found **159 stock, 6 macro and 12 COE** features (177 in total), so those counts are printed for comparison.

In [55]:
full = coverage[coverage['full']]
selected = {'stock': full.loc[full['source'] == 'stock', 'feature'].tolist()[2:],
            'macro': full.loc[full['source'] == 'macro', 'feature'].tolist(),
            'coe': full.loc[full['source'] == 'coe', 'feature'].tolist()[2:]}

counts = {name: len(features) for name, features in selected.items()}
reference = {'stock': 159, 'macro': 6, 'coe': 12}
print(f"stock {counts['stock']} | macro {counts['macro']} | COE {counts['coe']} | total {sum(counts.values())}")
print('matches the original pipeline:', counts == reference)

all_selected_features = []

for ft_module in selected:
    print(f'\n{ft_module}:', selected[ft_module])
    all_selected_features += selected[ft_module]


stock 159 | macro 6 | COE 12 | total 177
matches the original pipeline: True

stock: ['stk_ascendas_reit_log_price_mean', 'stk_ascendas_reit_mom_return', 'stk_ascendas_reit_return_volatility', 'stk_boc_aviation_log_price_mean', 'stk_boc_aviation_mom_return', 'stk_boc_aviation_return_volatility', 'stk_capitaland_ascott_trust_log_price_mean', 'stk_capitaland_ascott_trust_mom_return', 'stk_capitaland_ascott_trust_return_volatility', 'stk_capitaland_mall_trust_log_price_mean', 'stk_capitaland_mall_trust_mom_return', 'stk_capitaland_mall_trust_return_volatility', 'stk_city_developments_log_price_mean', 'stk_city_developments_mom_return', 'stk_city_developments_return_volatility', 'stk_cmon_log_price_mean', 'stk_cmon_mom_return', 'stk_cmon_return_volatility', 'stk_comfortdelgro_log_price_mean', 'stk_comfortdelgro_mom_return', 'stk_comfortdelgro_return_volatility', 'stk_dbs_group_log_price_mean', 'stk_dbs_group_mom_return', 'stk_dbs_group_return_volatility', 'stk_flex_log_price_mean', 'stk_fl

## 6. Save

In [58]:
temp_df = sources['stock']
temp_df = pd.concat([temp_df, sources['macro'].reset_index(drop = True)], axis = 1)
temp_df = pd.merge(temp_df, sources['coe'], how = 'left', on = ['RENT_APPROVAL_YEAR', 'RENT_APPROVAL_MONTH'])
temp_df = temp_df[['RENT_APPROVAL_YEAR', 'RENT_APPROVAL_MONTH']+all_selected_features]
temp_df

,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,stk_ascendas_reit_log_price_mean,stk_ascendas_reit_mom_return,stk_ascendas_reit_return_volatility,stk_boc_aviation_log_price_mean,stk_boc_aviation_mom_return,stk_boc_aviation_return_volatility,stk_capitaland_ascott_trust_log_price_mean,stk_capitaland_ascott_trust_mom_return,...,coe_a_quota_premium,coe_b_bids_received,coe_b_quota,coe_b_quota_premium,coe_c_bids_received,coe_c_quota,coe_c_quota_premium,coe_e_bids_received,coe_e_quota,coe_e_quota_premium
0,2021,1,0.806770,NaN,0.012960,3.925360,NaN,0.024428,-0.267930,NaN,...,42055.0,1367.0,1020.5,49550.5,675.0,376.5,38449.5,731.5,466.5,50694.5
1,2021,2,0.810418,0.003648,0.010391,3.962258,0.036898,0.024973,-0.313300,-0.045370,...,41500.5,1137.5,942.5,46396.0,467.0,267.5,42945.5,469.0,302.0,48642.0
2,2021,3,0.786484,-0.023934,0.012611,4.087016,0.124758,0.023987,-0.258145,0.055155,...,43292.5,1171.5,944.0,46001.0,493.5,273.5,38551.0,440.0,297.0,47403.5
3,2021,4,0.824610,0.038125,0.006253,4.034332,-0.052684,0.020803,-0.249994,0.008150,...,47620.0,1554.5,957.0,56749.5,464.5,262.5,40067.5,634.0,299.5,57150.0
4,2021,5,0.792135,-0.032474,0.012063,4.001513,-0.032818,0.016937,-0.306375,-0.056381,...,44901.5,1166.5,818.0,59045.0,288.0,160.5,42351.0,393.5,267.0,62501.0
5,2021,6,0.793423,0.001288,0.008370,3.983091,-0.018423,0.011150,-0.301866,0.004509,...,48165.5,1045.0,812.0,58070.5,445.5,216.5,37001.0,425.0,262.0,59556.5
6,2021,7,0.823071,0.029648,0.008517,3.900287,-0.082804,0.022748,-0.288807,0.013059,...,46005.5,1097.0,829.5,57800.5,338.5,164.0,39211.5,412.0,263.0,58850.5
7,2021,8,0.851118,0.028047,0.007502,3.860988,-0.039300,0.025141,-0.288279,0.000527,...,45939.0,1022.5,782.5,58501.0,226.5,118.0,41299.5,348.0,198.5,62250.0
8,2021,9,0.843065,-0.008053,0.009223,3.901717,0.040730,0.015466,-0.315526,-0.027246,...,47500.0,1134.5,781.5,65455.0,207.0,121.0,39500.5,359.5,202.0,67351.0
9,2021,10,0.825418,-0.017647,0.006720,4.000813,0.099095,0.017962,-0.304854,0.010672,...,49855.0,1185.0,774.0,75205.0,242.5,125.5,39695.5,416.5,212.0,78878.0


In [62]:
coverage.to_csv(f'{OUTPUT}/feature_coverage.csv', index=False)
with open(f'{OUTPUT}/selected_temporal_features.json', 'w') as f:
    json.dump(selected, f, indent=2)

temp_df.to_csv(f'{OUTPUT}/selected_temporal_features.csv', index = False)
print('saved to', OUTPUT)

saved to /Users/keltonn/Desktop/Projects/housing_rent_price/final_report/data/features/cleaned/temporal_features
